# Figure 3: Scaling Theory and Ensemble Predictions

Reproduces the Figure 3 scaling plots and overlays Wigner semicircle predictions.

In [ ]:
# Colab setup:
# from google.colab import drive
# drive.mount('/content/drive')
#
# import sys
# sys.path.insert(0, '/content/drive/MyDrive/phase_separation')
#
# !cp -r "/content/drive/MyDrive/phase_separation" /content/phase_separation
# %cd phase_separation
# !ls
# !pip install -r requirements.txt
# !pip install jax-tqdm scikit-learn

import os
import sys

sys.path.insert(0, os.path.abspath('../..'))

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import time

from jax_phase_separation.utils import (
    generate_chi_matrix, generate_initial_conditions, build_params,
)
from jax_phase_separation.solver import simulate
from jax_phase_separation.free_energy import compute_jacobian, stability_analysis
from jax_phase_separation.analysis import analyse_snapshot
from jax_phase_separation.theory import n_phases_wigner, n_phases_linear

print('JAX version:', jax.__version__)

In [ ]:
# Shared simulation settings
N_GRID = 64
DT = 5e-6
LMBDA = 0.01
N_STEPS = 2_000_000
N_REPS = 5  # replicates per condition


def run_condition(n_com, sigma, n_reps=N_REPS, beta=None, seed_offset=0):
    """Run multiple replicates and return (n_unstable_list, n_phases_list)."""
    if beta is None:
        beta = n_com / (n_com + 1.0)
    master = jax.random.PRNGKey(seed_offset)
    keys = jax.random.split(master, n_reps * 2).reshape(n_reps, 2, -1)

    n_unstable_list = []
    n_phases_list = []

    for rep in range(n_reps):
        chi = generate_chi_matrix(n_com, 0.0, sigma, keys[rep, 0])
        c0 = generate_initial_conditions(n_com, N_GRID, beta=beta,
                                         noise_strength=0.01, key=keys[rep, 1])
        params = build_params(chi, n_com, beta=beta, lmbda=LMBDA, dt=DT)

        chi_s_vec = jnp.zeros(n_com)
        r_vec = jnp.ones(n_com)
        J = compute_jacobian(n_com, beta, chi, chi_s_vec, r_vec)
        _, _, n_uns = stability_analysis(J)
        n_unstable_list.append(int(n_uns))

        c_final = simulate(c0, params, N_GRID, N_STEPS, progress_bar=True)
        res = analyse_snapshot(np.array(c_final))
        n_phases_list.append(res['n_phases'])

    return np.array(n_unstable_list), np.array(n_phases_list)

## Figure 3A: N_phases vs N_{lambda<0}

Run simulations across a range of (N, sigma) to build the collapse plot.

In [ ]:
conditions_3a = [
    (8,  3.5), (8,  4.8), (8,  5.5),
    (12, 4.0), (12, 5.2),
    (16, 4.8), (16, 5.5),
    (20, 5.0), (20, 5.4),
]

all_n_unstable = []
all_n_phases_3a = []

for i, (nc, sig) in enumerate(conditions_3a):
    print(f'  Condition {i+1}/{len(conditions_3a)}: N={nc}, sigma={sig} ...', end=' ')
    t0 = time.time()
    n_uns, n_ph = run_condition(nc, sig, n_reps=10, seed_offset=i * 1000)
    print(f'{time.time()-t0:.0f}s  phases={n_ph.mean():.1f}+/-{n_ph.std():.1f}')
    all_n_unstable.extend(n_uns.tolist())
    all_n_phases_3a.extend(n_ph.tolist())

all_n_unstable = np.array(all_n_unstable)
all_n_phases_3a = np.array(all_n_phases_3a)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

# Bin by n_unstable and plot mean +/- std
unique_n_uns = np.unique(all_n_unstable)
means, stds = [], []
for nu in unique_n_uns:
    mask = all_n_unstable == nu
    means.append(all_n_phases_3a[mask].mean())
    stds.append(all_n_phases_3a[mask].std())
means = np.array(means)
stds = np.array(stds)

ax.bar(unique_n_uns, means, yerr=stds, alpha=0.6, color='steelblue',
       edgecolor='k', capsize=3, label='simulation')
x_line = np.arange(0, int(unique_n_uns.max()) + 2)
ax.plot(x_line, x_line + 1, 'k-', lw=2, label='$N_{ph} = N_{\\lambda<0} + 1$')

ax.set_xlabel('$N_{\\lambda<0}$', fontsize=13)
ax.set_ylabel('$N_{phases}$', fontsize=13)
ax.set_title('Figure 3A', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Figures 3C,D: alpha-ensemble

In the alpha-ensemble, sigma = alpha * sqrt(N).  Varying N should give linear
scaling of N_phases.  Varying alpha shows saturation.

In [ ]:
alpha_values = [1.2, 1.5]
N_components_alpha = [4, 8, 12, 16, 20]

results_alpha = {}  # (alpha, N) -> (mean, std)

for alpha in alpha_values:
    for nc in N_components_alpha:
        sigma = alpha * np.sqrt(nc)
        print(f'  alpha={alpha}, N={nc}, sigma={sigma:.1f} ...', end=' ')
        t0 = time.time()
        _, n_ph = run_condition(nc, sigma, n_reps=10,
                                seed_offset=int(alpha * 1000 + nc))
        print(f'{time.time()-t0:.0f}s  phases={n_ph.mean():.1f}')
        results_alpha[(alpha, nc)] = (n_ph.mean(), n_ph.std())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 3C: N_phases vs N_components for different alpha
N_theory = np.linspace(2, 22, 100)
for alpha in alpha_values:
    Ns = np.array(N_components_alpha)
    ms = [results_alpha[(alpha, n)][0] for n in Ns]
    ss = [results_alpha[(alpha, n)][1] for n in Ns]
    ax1.errorbar(Ns, ms, yerr=ss, fmt='o-', capsize=3, label=f'sim $\\alpha$={alpha}')
    sigma_theory = alpha * np.sqrt(N_theory)
    ax1.plot(N_theory, n_phases_wigner(N_theory, sigma_theory), '--',
             label=f'theory $\\alpha$={alpha}')

ax1.set_xlabel('$N_{components}$', fontsize=12)
ax1.set_ylabel('$N_{phases}$', fontsize=12)
ax1.set_title('Figure 3C: $\\alpha$-ensemble', fontsize=13)
ax1.legend(fontsize=9)

# 3D: N_phases vs alpha for different N
N_vals_for_alpha = [12, 16, 20]
alpha_sweep = np.linspace(0.5, 2.0, 100)

for nc in N_vals_for_alpha:
    alphas_sim = [a for a in alpha_values if (a, nc) in results_alpha]
    ms = [results_alpha[(a, nc)][0] for a in alphas_sim]
    ss = [results_alpha[(a, nc)][1] for a in alphas_sim]
    ax2.errorbar(alphas_sim, ms, yerr=ss, fmt='o', capsize=3, label=f'sim N={nc}')
    sigma_theory = alpha_sweep * np.sqrt(nc)
    ax2.plot(alpha_sweep, n_phases_wigner(nc, sigma_theory), '--',
             label=f'theory N={nc}')

ax2.set_xlabel('$\\alpha$', fontsize=12)
ax2.set_ylabel('$N_{phases}$', fontsize=12)
ax2.set_title('Figure 3D', fontsize=13)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Figures 3E,F: sigma-ensemble (constant sigma)

Fixed sigma leads to nonmonotonic scaling of N_phases with N_components.

In [ ]:
sigma_values = [3.0, 4.0, 5.0]
N_components_sigma = [4, 8, 12, 16, 20]

results_sigma = {}

for sigma in sigma_values:
    for nc in N_components_sigma:
        print(f'  sigma={sigma}, N={nc} ...', end=' ')
        t0 = time.time()
        _, n_ph = run_condition(nc, sigma, n_reps=10,
                                seed_offset=int(sigma * 1000 + nc))
        print(f'{time.time()-t0:.0f}s  phases={n_ph.mean():.1f}')
        results_sigma[(sigma, nc)] = (n_ph.mean(), n_ph.std())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 3E: N_phases vs N_components for different sigma
N_theory = np.linspace(2, 22, 100)
for sigma in sigma_values:
    Ns = np.array(N_components_sigma)
    ms = [results_sigma[(sigma, n)][0] for n in Ns]
    ss = [results_sigma[(sigma, n)][1] for n in Ns]
    ax1.errorbar(Ns, ms, yerr=ss, fmt='o-', capsize=3, label=f'sim $\\sigma$={sigma}')
    ax1.plot(N_theory, n_phases_wigner(N_theory, sigma), '--',
             label=f'theory $\\sigma$={sigma}')

ax1.set_xlabel('$N_{components}$', fontsize=12)
ax1.set_ylabel('$N_{phases}$', fontsize=12)
ax1.set_title('Figure 3E: $\\sigma$-ensemble', fontsize=13)
ax1.legend(fontsize=9)

# 3F: N_phases vs sigma for different N
N_vals_for_sigma = [12, 16, 20]
sigma_sweep = np.linspace(1.0, 7.0, 100)

for nc in N_vals_for_sigma:
    sigmas_sim = [s for s in sigma_values if (s, nc) in results_sigma]
    ms = [results_sigma[(s, nc)][0] for s in sigmas_sim]
    ss = [results_sigma[(s, nc)][1] for s in sigmas_sim]
    ax2.errorbar(sigmas_sim, ms, yerr=ss, fmt='o', capsize=3, label=f'sim N={nc}')
    ax2.plot(sigma_sweep, n_phases_wigner(nc, sigma_sweep), '--',
             label=f'theory N={nc}')

ax2.set_xlabel('$\\sigma$', fontsize=12)
ax2.set_ylabel('$N_{phases}$', fontsize=12)
ax2.set_title('Figure 3F', fontsize=13)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()